# M9.10 · Frekans uzayı — DFT, filtreler ve ters süzgeçleme

Doğrusal dönüşümler ve dönüşüm çiftleri, ayrılabilir ve simetrik çekirdekler, LTI sistemlerin özfonksiyonu olarak karmaşık üsteller, konvolüsyon teoremi ve sarmalanma (wraparound). Elle küçük DFT'ler ve dairesel konvolüsyon; saf Python'da DFT ve Cooley-Tukey FFT; NumPy'da Gonzalez'in dolgulu frekans filtreleme hattı, ideal / Butterworth / Gauss süzgeçleri ve FFT ile doğrusal konvolüsyon; PyTorch'ta bulanık ve gürültülü görüntülerden frekans başına kazancı öğrenip Wiener süzgecine ulaşıyorsun; en sonda `F.conv2d` ile aynı sonucu veren FFT tabanlı konvolüsyonu yazıp büyük çekirdeklerde ne zaman kazandığını ölçüyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m9-10-frequency.html](https://m-gonul.github.io/pytorch-lab/m9-10-frequency.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m9_10_frequency.py
CHECKS = {
 "b": "import time\nimport numpy as np\nfrom labcheck import close as _close, no_imports, need, ensure\n\n\ndef close(a, b, **kw):\n    a, b = np.asarray(a, np.complex128), np.asarray(b, np.complex128)\n    _close(a.real, b.real, **kw)\n    _close(a.imag, b.imag, **kw)\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"scipy\", \"torch\"}))\n    rng = np.random.default_rng(0)\n\n    def d():\n        for N in (1, 5, 8, 12):\n            x = rng.normal(size=N)\n            close(np.array(need(ns, \"dft\")(x.tolist())), np.fft.fft(x), atol=1e-9, what=f\"N = {N}\")\n    R.test(\"dft = np.fft.fft\", d)\n\n    def f():\n        for N in (2, 16, 64, 256, 6):\n            x = rng.normal(size=N) + 1j * rng.normal(size=N)\n            close(np.array(need(ns, \"fft\")(x.tolist())), np.fft.fft(x), atol=1e-8, what=f\"N = {N}\")\n            close(np.array(need(ns, \"idft\")(np.fft.fft(x).tolist())), x, atol=1e-9, what=f\"idft N = {N}\")\n    R.test(\"fft ve idft\", f)\n\n    def two():\n        img = rng.normal(size=(8, 16))\n        close(np.array(need(ns, \"dft2\")(img.tolist())), np.fft.fft2(img), atol=1e-8, what=\"dft2\")\n    R.test(\"dft2 = np.fft.fft2 (ayrılabilirlik)\", two)\n\n    def speed():\n        x = rng.normal(size=1024).tolist()\n        t0 = time.perf_counter(); ns[\"dft\"](x[:256]); t1 = time.perf_counter(); ns[\"fft\"](x); t2 = time.perf_counter()\n        R.table(\"Saf Python\", [\"işlem\", \"N\", \"süre\", \"çarpma sayısı\"], [[\"dft\", 256, f\"{(t1 - t0) * 1e3:.0f} ms\", f\"{256 ** 2:,}\"], [\"fft\", 1024, f\"{(t2 - t1) * 1e3:.0f} ms\", f\"≈ {512 * 10:,}\"]],\n                note=\"FFT: N/2 · log₂N kelebek çarpması. 1024 noktada DFT'nin 1/200'ü.\")\n    R.test(\"O(N²) ve O(N log N)\", speed)\n",
 "c": "import numpy as np\nfrom scipy import signal\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n\n    def hp():\n        H = need(ns, \"lowpass\")(\"butterworth\", (64, 80), 10, 2)\n        close(H[32, 40], 1.0, what=\"merkez\"); close(H[42, 40], 0.5, what=\"D = D0\")\n        close(need(ns, \"lowpass\")(\"gaussian\", (64, 80), 10)[32, 50], np.exp(-0.5), what=\"Gauss D = D0\")\n        I = ns[\"lowpass\"](\"ideal\", (64, 80), 10)\n        ensure(I[32, 50] == 1 and I[32, 51] == 0, \"ideal sınır dahil\")\n        close(need(ns, \"highpass\")(\"gaussian\", (64, 80), 10), 1 - ns[\"lowpass\"](\"gaussian\", (64, 80), 10), what=\"yüksek geçiren\")\n    R.test(\"süzgeç tasarımı\", hp)\n\n    def ff():\n        img = data.camera()[::2, ::2].astype(np.float64)\n        M, N = img.shape\n        H = ns[\"lowpass\"](\"gaussian\", (2 * M, 2 * N), 25)\n        got = need(ns, \"freq_filter\")(img, H)\n        fp = np.zeros((2 * M, 2 * N)); fp[:M, :N] = img\n        want = np.real(np.fft.ifft2(np.fft.ifftshift(H) * np.fft.fft2(fp)))[:M, :N]\n        close(got, want, atol=1e-8, what=\"Gonzalez hattı = fftshift'li hat\")\n    R.test(\"freq_filter\", ff)\n\n    def cf():\n        rng = np.random.default_rng(0)\n        img = rng.random((60, 70)); k = rng.random((9, 5))\n        close(need(ns, \"conv_fft\")(img, k), signal.convolve2d(img, k, \"full\"), atol=1e-9, what=\"tam konvolüsyon\")\n    R.test(\"conv_fft = scipy convolve2d (full)\", cf)\n\n    def ring():\n        img = np.zeros((128, 128)); img[:, 64:] = 200\n        H = lambda kind: ns[\"lowpass\"](kind, (256, 256), 15, 2)\n        rows = []\n        over = {}\n        for kind in (\"ideal\", \"butterworth\", \"gaussian\"):\n            out = ns[\"freq_filter\"](img, H(kind))[64, 20:110]\n            over[kind] = max(out.max() - 200, 0)\n            rows.append([kind, f\"{out.max():.1f}\", f\"{over[kind]:.1f}\"])\n        ensure(over[\"ideal\"] > over[\"butterworth\"] > over[\"gaussian\"] - 1e-9, f\"aşma: {over}\")\n        R.table(\"Basamak kenarı, D0 = 15\", [\"süzgeç\", \"en büyük değer (basamak 200)\", \"aşma\"], rows, note=\"İdeal süzgecin sinc yanıtı halkalanır; n = 2 Butterworth hafifçe; Gauss hiç.\")\n    R.test(\"halkalanma: ideal > Butterworth > Gauss\", ring)\n",
 "d": "import numpy as np\nimport torch\nfrom skimage import data, color\nfrom labcheck import close, need, ensure\n\nN = 128\n\n\ndef tiles():\n    out = []\n    for im in [data.camera(), color.rgb2gray(data.astronaut()) * 255, color.rgb2gray(data.coffee()) * 255, data.moon()]:\n        im = im.astype(np.float32) / 255\n        for i in range(0, im.shape[0] - N + 1, N):\n            for j in range(0, im.shape[1] - N + 1, N):\n                out.append(im[i:i + N, j:j + N])\n    return torch.tensor(np.stack(out))\n\n\ndef check(ns, R):\n    def otf():\n        H = need(ns, \"gaussian_otf\")(N, 2.0)\n        ensure(tuple(H.shape) == (N, N // 2 + 1), \"rfft2 düzeni\")\n        psf = np.zeros((N, N)); r = np.arange(N); r = np.minimum(r, N - r)\n        psf = np.exp(-(r[:, None] ** 2 + r[None, :] ** 2) / 8.0); psf /= psf.sum()\n        close(H.double().numpy(), np.real(np.fft.rfft2(psf)), atol=1e-4, what=\"dairesel Gauss PSF'nin DFT'si\")\n    R.test(\"gaussian_otf\", otf)\n\n    def rest():\n        H = ns[\"gaussian_otf\"](N, 2.0).float()\n        train = tiles()\n        test = torch.tensor(data.coins()[:N, :N].astype(np.float32) / 255)\n        g = torch.Generator().manual_seed(1)\n        y = need(ns, \"degrade\")(test, H, 0.01, g)\n        P = lambda a: 10 * torch.log10(1 / ((a - test) ** 2).mean()).item()\n        inv = P(need(ns, \"apply\")(1 / H, y))\n        wk = {K: P(ns[\"apply\"](need(ns, \"wiener\")(H, K), y)) for K in (1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1)}\n        Kb = max(wk, key=wk.get)\n        G = need(ns, \"learn_filter\")(train, H, 0.01)\n        pl = P(ns[\"apply\"](G, y))\n        ensure(pl > wk[Kb] - 0.3, f\"öğrenilen {pl:.2f} dB, en iyi Wiener {wk[Kb]:.2f} dB\")\n        ensure(pl > P(y) + 1.0, \"bulanık girdiden en az 1 dB iyi olmalı\")\n        R.table(\"coins (eğitimde yok), Gauss σ = 2 bulanıklık + σ = 0.01 gürültü\", [\"yöntem\", \"PSNR (dB)\"],\n                [[\"bulanık + gürültülü\", f\"{P(y):.2f}\"], [\"ters süzgeç 1/H\", f\"{inv:.1f}\"], [f\"Wiener (en iyi K = {Kb:g})\", f\"{wk[Kb]:.2f}\"], [\"öğrenilen G (600 adım)\", f\"{pl:.2f}\"]],\n                note=\"Ters süzgeç, H ≈ e⁻²⁰⁰ olan frekanslarda gürültüyü astronomik ölçüde büyütür.\")\n        prof = G[0, :40].numpy(); wp = ns[\"wiener\"](H, Kb)[0, :40].numpy()\n        c = float(np.corrcoef(prof, wp)[0, 1])\n        ensure(c > 0.9, f\"öğrenilen kazanç profili Wiener'e benzememiş (korelasyon {c:.2f})\")\n    R.test(\"öğrenilen frekans süzgeci ≈ Wiener\", rest)\n",
 "e": "import torch\nimport torch.nn.functional as F\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    g = torch.Generator().manual_seed(0)\n    for (B, Ci, Co, H, W, k, p) in [(2, 3, 4, 20, 17, 3, 1), (1, 2, 5, 32, 32, 7, 0), (3, 1, 1, 15, 40, 9, 4), (1, 4, 2, 24, 24, (3, 11), 0)]:\n        def t(B=B, Ci=Ci, Co=Co, H=H, W=W, k=k, p=p):\n            kh, kw = (k, k) if isinstance(k, int) else k\n            x = torch.randn(B, Ci, H, W, generator=g, dtype=torch.float64)\n            w = torch.randn(Co, Ci, kh, kw, generator=g, dtype=torch.float64)\n            close(need(ns, \"conv2d_fft\")(x, w, p), F.conv2d(x, w, padding=p), atol=1e-9, what=\"F.conv2d\")\n        R.test(f\"= F.conv2d (x {B}×{Ci}×{H}×{W}, w {Co}×{Ci}×{k}, padding {p})\", t)\n    if \"conv2d_fft\" not in ns:\n        return\n    rows = []\n    x = torch.randn(1, 8, 128, 128)\n    for k in (3, 7, 15, 31, 63):\n        w = torch.randn(8, 8, k, k)\n        t1 = bench(lambda: F.conv2d(x, w, padding=k // 2), repeat=3)\n        t2 = bench(lambda: ns[\"conv2d_fft\"](x, w, k // 2), repeat=3)\n        rows.append([k, fmt_time(t1), fmt_time(t2), \"FFT\" if t2 < t1 else \"doğrudan\"])\n    R.table(\"1 × 8 × 128 × 128 → 8 kanal, 'same' dolgu\", [\"k\", \"F.conv2d\", \"conv2d_fft\", \"hızlı olan\"], rows,\n            note=\"Doğrudan konvolüsyon k² ile, FFT neredeyse sabit büyür; kesişim donanıma ve kütüphaneye bağlıdır.\")\n",
 "x1": "import numpy as np\nfrom skimage import data, color\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    a = data.camera().astype(np.float64)\n    b = (color.rgb2gray(data.astronaut()) * 255)\n\n    def sw():\n        out = need(ns, \"swap_phase\")(a, b)\n        c = lambda x, y: float(np.corrcoef(x.ravel(), y.ravel())[0, 1])\n        ca, cb = c(out, a), c(out, b)\n        ensure(cb > ca + 0.2, f\"faz b'den: b ile korelasyon {cb:.2f}, a ile {ca:.2f}\")\n        return f\"|F_camera| + ∠F_astronaut → astronaut ile korelasyon {cb:.2f}, camera ile {ca:.2f}: yapıyı faz taşır\"\n    R.test(\"swap_phase\", sw)\n\n    def sp():\n        s = need(ns, \"spectrum_log\")(a)\n        ensure(s.dtype == np.uint8 and s.max() == 255 and s.min() == 0, \"0–255 uint8\")\n        ensure(np.unravel_index(np.argmax(s), s.shape) == (256, 256), \"DC merkezde olmalı\")\n    R.test(\"spectrum_log merkezli\", sp)\n\n    def sh():\n        close(need(ns, \"shift\")(a, 7, -3), np.roll(a, (-3, 7), (0, 1)), atol=1e-8, what=\"tamsayı kayma = np.roll\")\n        o = a[:511, :511]\n        h = ns[\"shift\"](ns[\"shift\"](o, 0.5, 0.25), 0.5, 0.75)\n        close(h, np.roll(o, (1, 1), (0, 1)), atol=1e-6, what=\"iki kesirli kayma = (1, 1) (tek boyutlu görüntü: Nyquist bileşeni yok)\")\n    R.test(\"öteleme teoremi\", sh)\n",
 "x2": "import numpy as np\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    img = data.camera().astype(np.float64)\n    M, N = img.shape\n    y, x = np.mgrid[:M, :N]\n    noise = 40 * np.sin(2 * np.pi * (30 * y / M + 50 * x / N)) + 30 * np.sin(2 * np.pi * (-70 * y / M + 20 * x / N))\n    noisy = img + noise\n\n    def sp():\n        got = set(need(ns, \"find_spikes\")(noisy, 4))\n        ensure(got == {(30, 50), (-30, -50), (-70, 20), (70, -20)}, f\"tepeler: {sorted(got)}\")\n    R.test(\"find_spikes (iki sinüs = dört tepe)\", sp)\n\n    def nr():\n        H = need(ns, \"notch_reject\")((64, 64), [(5, 3)], 2)\n        ensure(H[32 + 5, 32 + 3] == 0 and H[32 - 5, 32 - 3] == 0, \"çentik merkezleri sıfır\")\n        close(H[0, 0], 1.0, atol=0.01, what=\"uzakta ≈ 1\")\n    R.test(\"notch_reject\", nr)\n\n    def rp():\n        out = need(ns, \"remove_periodic\")(noisy, 2, 3)\n        P = lambda a: 10 * np.log10(255 ** 2 / np.mean((a - img) ** 2))\n        ensure(P(out) > P(noisy) + 15, f\"PSNR {P(noisy):.1f} → {P(out):.1f}\")\n        return f\"PSNR {P(noisy):.1f} → {P(out):.1f} dB\"\n    R.test(\"periyodik gürültü temizleme\", rp)\n",
 "x3": "import numpy as np\nfrom scipy import fft\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    rng = np.random.default_rng(0)\n\n    def mat():\n        C = need(ns, \"dct_matrix\")(8)\n        close(C @ C.T, np.eye(8), atol=1e-12, what=\"ortonormal\")\n        X = rng.random((8, 12))\n        close(need(ns, \"dct2\")(X), fft.dctn(X, norm=\"ortho\"), atol=1e-12, what=\"dctn\")\n        close(need(ns, \"idct2\")(ns[\"dct2\"](X)), X, atol=1e-12, what=\"geri\")\n    R.test(\"DCT-II = scipy.fft.dctn (ortho)\", mat)\n\n    def comp():\n        img = data.camera()[::2, ::2]\n        P = lambda a: 10 * np.log10(255 ** 2 / np.mean((a - img) ** 2))\n        rows = []\n        for frac in (0.01, 0.05, 0.2):\n            d = P(need(ns, \"keep_largest\")(img, frac, \"dct\")); f = P(ns[\"keep_largest\"](img, frac, \"dft\"))\n            rows.append([f\"%{frac * 100:g}\", f\"{d:.2f}\", f\"{f:.2f}\"])\n            ensure(d > f, f\"%{frac * 100:g} katsayıda DCT {d:.2f} dB, DFT {f:.2f} dB\")\n        R.table(\"En büyük katsayıları tutma (camera 256²)\", [\"katsayı oranı\", \"DCT PSNR\", \"DFT PSNR\"], rows,\n                note=\"DFT bir karmaşık katsayı = iki gerçel sayı olduğu için adil karşılaştırmada fark daha da büyür.\")\n    R.test(\"enerji yoğunlaştırma: DCT > DFT\", comp)\n"
}

In [ ]:
import importlib, labcheck, checks_m9_10_frequency as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Dönüşüm çifti:** $T(u,v)=\sum_{x,y}f(x,y)\,r(x,y,u,v)$, $f(x,y)=\sum_{u,v}T(u,v)\,s(x,y,u,v)$. Çekirdek **ayrılabilir** ise $r=r_1(x,u)r_2(y,v)$, **simetrik** ise $r_1=r_2$: 2-B dönüşüm 1-B dönüşümlerle (önce satırlar, sonra sütunlar) hesaplanır. DFT çekirdeği ikisi de:

$$F(u,v)=\sum_{x=0}^{M-1}\sum_{y=0}^{N-1}f(x,y)\,e^{-j2\pi(ux/M+vy/N)},\qquad f(x,y)=\frac1{MN}\sum_{u,v}F(u,v)\,e^{j2\pi(ux/M+vy/N)}.$$

**Neden frekans?** LTI/LSI sistemler konvolüsyonla tanımlanır ve karmaşık üsteller onların **özfonksiyonlarıdır**: $T[e^{j\omega t}]=H(\omega)e^{j\omega t}$. Bir sinyali üstellere ayırınca (Fourier = üstellere izdüşüm) sistemin etkisi frekans başına çarpmaya iner: **$f*h\Leftrightarrow F\cdot H$** (konvolüsyon teoremi). Bedeli: uzaysal konum bilgisi spektrumda görünmez, fazda saklıdır.

**Ayrık dünyanın tuzağı:** DFT periyodiktir, çarpım **dairesel** konvolüsyondur. Sarmalanmayı önlemek için iki diziyi $P\ge A+B-1$'e sıfırla doldur.

**Gonzalez hattı:** (1) $P=2M$, $Q=2N$ dolgu, (2) $(-1)^{x+y}$ ile merkezle, (3) DFT, (4) $G=H\cdot F$ (merkezli, simetrik $H$), (5) ters DFT, gerçek kısım, (6) $(-1)^{x+y}$, (7) sol üst $M\times N$'i kırp.

**Süzgeçler** ($D$: merkeze uzaklık): ideal $H=[D\le D_0]$ (halkalanır), Butterworth $\frac1{1+(D/D_0)^{2n}}$, Gauss $e^{-D^2/2D_0^2}$; yüksek geçiren $1-H_{LP}$.

**Restorasyon:** $G=HF+N$. Ters süzgeç $\hat F=G/H=F+N/H$: $H\to0$ olan yerde gürültü patlar. Wiener (en küçük ortalama kare): $\hat F=\frac{H^*}{|H|^2+S_\eta/S_f}G$, pratikte $S_\eta/S_f\approx K$ sabit.

## 1 · Elle

<!-- çözüm -->

$F(1)=1-2j-3+4j=-2+2j$, $F(2)=1-2+3-4=-2$.

Sarmalanma: $[1,0,0,2]$ ile $[1,1,0,0]$'nın doğrusal konvolüsyonu $[1,1,0,2,2]$; 4 uzunlukta dairesel alınca son eleman başa biner: $1+2=3$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**DFT, $f=[1,2,3,4]$, $F(u)=\sum_x f(x)e^{-j2\pi ux/4}$**

- `f0`: $F(0)$
- `f1r`: $\mathrm{Re}\,F(1)$
- `f1i`: $\mathrm{Im}\,F(1)$
- `f2`: $F(2)$

**Konvolüsyon ve dolgu**

- `cc`: Dairesel $[1,2,0,0]\circledast[1,1,0,0]$'nin 2. elemanı (indis 1)
- `wrap`: Dairesel $[1,0,0,2]\circledast[1,1,0,0]$'nin 1. elemanı (indis 0) — sarmalanma
- `pad`: 256 × 256 görüntü, 31 × 31 çekirdek: sarmalanmasız en küçük $P$

**Süzgeç kazançları**

- `bw`: Butterworth, $D=D_0$
- `gs`: Gauss, $D=D_0$
- `inv`: Ters süzgeç kazancı, $H=0.1$
- `wn`: Wiener kazancı $\frac{H^*}{|H|^2+K}$, $H=0.1$, $K=0.01$

In [ ]:
cevaplar = dict(
    f0=None,
    f1r=None,
    f1i=None,
    f2=None,
    cc=None,
    wrap=None,
    pad=None,
    bw=None,
    gs=None,
    inv=None,
    wn=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"f0":[10,0],"f1r":[-2,0],"f1i":[2,0],"f2":[-2,0],"cc":[3,0],"wrap":[3,0],"pad":[286,0],"bw":[0.5,0],"gs":[0.606531,0.0001],"inv":[10,0],"wn":[5,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python (`cmath` serbest), karmaşık sayı listeleri.

- `dft(x)` — tanımdan, $O(N^2)$.
- `fft(x)` — özyinelemeli Cooley-Tukey: $X_k=E_k+e^{-j2\pi k/N}O_k$, $X_{k+N/2}=E_k-e^{-j2\pi k/N}O_k$; $N$ 2'nin kuvveti değilse `dft`'ye düş.
- `idft(X)` — $\frac1N\sum X_ue^{+j2\pi ux/N}$ (ipucu: eşlenik, `fft`, eşlenik, $/N$).
- `dft2(img)` — ayrılabilirlik: her satıra, sonra her sütuna `fft`.

In [ ]:
%%writefile stage_b.py
import cmath


def dft(x):
    ...


def fft(x):
    ...


def idft(X):
    ...


def dft2(img):
    ...


print([round(abs(v), 3) for v in fft([1, 2, 3, 4])])

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok**.

- `lowpass(kind, shape, D0, n=2)` — merkezli $H$, `shape = (P, Q)`, $D(u,v)=\sqrt{(u-P/2)^2+(v-Q/2)^2}$ (tamsayı merkez `P//2`, `Q//2`); `kind` ∈ `"ideal"`, `"butterworth"`, `"gaussian"`. `highpass = 1 - lowpass` (ayrı fonksiyon).
- `freq_filter(img, H)` — Gonzalez'in 7 adımı; `H` şekli $(2M, 2N)$; `float64` döndür.
- `conv_fft(img, k)` — **tam** doğrusal konvolüsyon, `scipy.signal.convolve2d(img, k, "full")` ile aynı; `np.fft.rfft2(…, s=…)` ile.

In [ ]:
%%writefile stage_c.py
import numpy as np


def lowpass(kind, shape, D0, n=2):
    ...


def highpass(kind, shape, D0, n=2):
    ...


def freq_filter(img, H):
    ...


def conv_fft(img, k):
    ...


from skimage import data
img = data.camera()
print(freq_filter(img, lowpass("gaussian", (1024, 1024), 30)).shape)

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Bulanıklık + gürültü: $y=h\circledast x+\eta$ (dairesel, böylece $H$ DFT'de tam köşegen), $\sigma_\eta=0.01$.

- `gaussian_otf(N, s)` — $N\times N$ görüntü, $\sigma=s$ piksellik dairesel Gauss PSF'nin `rfft2` düzenindeki OTF'si: $e^{-2\pi^2s^2(u^2+v^2)/N^2}$, $u$ = `fftfreq(N)·N`, $v$ = `rfftfreq(N)·N`. `(N, N//2+1)`.
- `degrade(x, H, sigma, g)` — `irfft2(rfft2(x)·H) + sigma·randn` (`g` üreteciyle).
- `apply(G, y)` — `irfft2(rfft2(y)·G)`.
- `wiener(H, K)` — $\frac{H}{H^2+K}$ ($H$ gerçel).
- `learn_filter(train, H, sigma, steps=600, batch=16, seed=0)` — 1'lerle başlayan gerçel `G` (`H` ile aynı şekil); her adımda rastgele `batch` görüntü, `degrade`, MSE(`apply(G, y)`, `x`), Adam (lr = 1e-2). `G`'yi döndür.

Doğrusal bir onarıcının MSE'yi en aza indiren hali Wiener'dir; öğrenilen `G` ona (hatta gerçek spektrumu bildiği için sabit-$K$ Wiener'in biraz ötesine) yakınsar.

In [ ]:
%%writefile stage_d.py
import numpy as np
import torch


def gaussian_otf(N, s):
    ...


def degrade(x, H, sigma, g):
    ...


def apply(G, y):
    ...


def wiener(H, K):
    ...


def learn_filter(train, H, sigma, steps=600, batch=16, seed=0):
    ...


print(gaussian_otf(8, 1.0).shape)

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`conv2d_fft(x, w, padding=0)` — `F.conv2d(x, w, padding=padding)` ile aynı (yani **çapraz korelasyon**), `x (B, Cin, H, W)`, `w (Cout, Cin, kh, kw)`, `groups=1`, `stride=1`.

1. `x`'i `padding` kadar sıfırla doldur → $H_p\times W_p$.
2. Boyut $(H_p+k_h-1, W_p+k_w-1)$'de `rfft2(x)` ve `rfft2(w.flip(-2, -1))` (korelasyon = ters çevrilmiş çekirdekle konvolüsyon).
3. Kanallar üzerinden topla: `einsum("bihw,oihw->bohw")`.
4. `irfft2`, geçerli bölge: `[kh-1 : kh-1+Hout, kw-1 : kw-1+Wout]`.

Karmaşıklık: doğrudan $O(HWk^2)$, FFT $O(HW\log HW)$ — kesişim noktasını ölç.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F


def conv2d_fft(x, w, padding=0):
    ...


x = torch.randn(1, 2, 16, 16); w = torch.randn(3, 2, 5, 5)
print((conv2d_fft(x, w, 2) - F.conv2d(x, w, padding=2)).abs().max())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Spektrum ve faz: hangisi yapıyı taşır? Öteleme teoremi

- `swap_phase(a, b)` — $|F_a|$ ve $\angle F_b$'den görüntü: `np.real(ifft2(|Fa|·exp(j∠Fb)))`.
- `spectrum_log(img)` — merkezli `log(1+|F|)`, $[0,255]$'e ölçekli `uint8` (görselleştirme).
- `shift(img, dx, dy)` — öteleme teoremi: $f(x-x_0,y-y_0)\Leftrightarrow F(u,v)e^{-j2\pi(ux_0/M+vy_0/N)}$ ile dairesel kaydırma (tamsayı kayma `np.roll` ile aynı, kesirli kayma da çalışır). Gerçek kısmı döndür. Çift boyutta Nyquist frekansının kesirli kayması belirsizdir (gerçek kısım alınca bilgi kaybolur); kontrol tek boyutlu görüntü kullanır.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def swap_phase(a, b):
    ...


def spectrum_log(img):
    ...


def shift(img, dx, dy):
    ...


from skimage import data
print(spectrum_log(data.camera()).max())

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Periyodik gürültü ve çentik (notch) süzgeçleri

Periyodik gürültü (ör. elektrik paraziti) spektrumda **simetrik tepe çiftleri** olarak görünür.

- `find_spikes(img, k, r_dc=10)` — merkezli $|F|$'de DC çevresindeki `r_dc` yarıçapı dışında en büyük `k` noktanın `(u, v)` (merkezli koordinatlar, `(satır, sütun)`) listesi.
- `notch_reject(shape, centers, D0, n=2)` — Butterworth çentik reddetme: $\prod_k\frac1{1+(D_0/D_k)^{2n}}\cdot\frac1{1+(D_0/D_{-k})^{2n}}$; $D_k$ merkezli spektrumda $(u_k, v_k)$'ye, $D_{-k}$ simetriğine uzaklık.
- `remove_periodic(img, k, D0)` — `find_spikes` (tepeleri çift olarak bulur; her çiftten birini merkez al) + `notch_reject`, **dolgusuz** (gürültü zaten periyodik) merkezli spektrumda süzgeçle.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def find_spikes(img, k, r_dc=10):
    ...


def notch_reject(shape, centers, D0, n=2):
    ...


def remove_periodic(img, k, D0):
    ...


print(notch_reject((32, 32), [(5, 3)], 2).min())

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · DCT ve enerji yoğunlaştırma (JPEG'in temeli)

- `dct_matrix(N)` — ortonormal DCT-II: $C_{kn}=\alpha_k\cos\frac{\pi(2n+1)k}{2N}$, $\alpha_0=\sqrt{1/N}$, $\alpha_k=\sqrt{2/N}$.
- `dct2(X)` — $CXC^T$ (`scipy.fft.dctn(…, norm="ortho")` ile aynı) ve `idct2(Y)`.
- `keep_largest(img, frac, kind)` — `kind` `"dct"` ya da `"dft"`: tüm görüntünün dönüşümünde en büyük genlikli `frac` oranında katsayıyı tut, gerisini sıfırla, geri dön (DFT'de gerçek kısım).

DCT, simetrik uzantı yaptığı için kenarlarda sıçrama üretmez; aynı katsayı bütçesinde DFT'den yüksek PSNR verir.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def dct_matrix(N):
    ...


def dct2(X):
    ...


def idct2(Y):
    ...


def keep_largest(img, frac, kind):
    ...


print(np.round(dct_matrix(4) @ dct_matrix(4).T, 6))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Global Filter Network: frekans uzayında öğrenilen küresel süzgeçler

[Rao, Zhao, Zhu, Lu, Zhou — Global Filter Networks for Image Classification (NeurIPS 2021)](https://arxiv.org/abs/2107.00645)

ViT'teki öz-dikkat katmanını, yama ızgarası üzerinde 2-B FFT → öğrenilen karmaşık süzgeçle eleman eleman çarpım → ters FFT işlemiyle değiştiren GFNet bloğunu uygula. CIFAR-10'da aynı derinlik ve genişlikteki küçük bir ViT ile doğruluk, parametre ve FLOP karşılaştır; öğrenilen süzgeçleri görselleştir.


**Adımlar**

1. Bu modüldeki d aşamasının frekans başına öğrenilen kazancını kanal başına karmaşık süzgece genelleştir (`torch.fft.rfft2`).
2. Blok: LN → global filter → LN → MLP, artık bağlantılarla; yama gömme M4.15'teki gibi.
3. Görüntü çözünürlüğü değişince süzgeci enterpolasyonla yeniden boyutlandır.
4. Karmaşıklık: öz-dikkat $O(N^2)$, global filter $O(N\log N)$; N ile süre ölçümü.

**Kabul ölçütleri**

- CIFAR-10'da GFNet ve ViT doğrulukları ±2 puan içinde; GFNet FLOP'ları daha düşük.
- Token sayısı 64 → 1024 için süre grafiği.
- Öğrenilen süzgeçlerin genlik spektrumları: sığ katmanlarda yüksek frekans ağırlıklı mı?

## Kaynaklar

- [Gonzalez & Woods — Digital Image Processing, Bölüm 4 ve 5.7–5.8](https://www.imageprocessingplace.com/) — ©
- [Cooley & Tukey — An algorithm for the machine calculation of complex Fourier series (1965)](https://doi.org/10.1090/S0025-5718-1965-0178586-1) — ©
- [PyTorch — torch.fft](https://pytorch.org/docs/stable/fft.html) — BSD-3
- [Szeliski — Computer Vision: Algorithms and Applications, 2. baskı, 3.4](https://szeliski.org/Book/) — ücretsiz PDF